In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("risk_analytics_train.csv")
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

# imputing categorical data with mode value

In [ ]:
for value in ['Gender', 'Married', 'Dependents','Self_Employed', 'Credit_History']:
    df[value].fillna(df[value].mode()[0],inplace=True)

In [ ]:
df.isnull().sum()

# imputing numerical data with mean value

In [ ]:
col= ['LoanAmount','Loan_Amount_Term']
for i in col:
    df[i].fillna(round(df[i].mean(),0),inplace=True)

In [ ]:
df.isnull().sum()

# Transforming categorical data into numerical data

In [ ]:
from sklearn.preprocessing import LabelEncoder
colname=['Gender', 'Married', 'Education','Self_Employed','Property_Area', 'Loan_Status']
le=LabelEncoder()
for x in colname:
    df[x]=le.fit_transform(df[x])

In [ ]:
df

In [ ]:
corr_df = df.drop('Loan_ID', axis=1).corr()
corr_df

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(corr_df,vmin=-1.0,vmax=1.0,cmap='plasma',annot=True)
plt.show()

In [ ]:
X=df.drop(['Loan_Status','Loan_ID'],axis=1)
Y=df.Loan_Status

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(X)
x=scaler.transform(X)

In [ ]:
x

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,Y,test_size=0.2,random_state=10)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
y_test.shape

In [ ]:
from sklearn.svm import SVC
svc_model=SVC(kernel='rbf',C=10,gamma=0.002)
svc_model.fit(X_train,y_train)

In [ ]:
y_pred=svc_model.predict(X_test)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [ ]:
acc=accuracy_score(y_test,y_pred)
print("accuracy_score",acc*100)

In [ ]:
con=confusion_matrix(y_test,y_pred)
print(con)

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV
dict1={'kernel':['rbf','poly'],'C':[5,10,15,20],'gamma':[0.1,0.2,0.3,0.4]}
grid_search = GridSearchCV(estimator=svc_model, param_grid=dict1, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
best_params = grid_search.best_params_
svc_tuned_model = SVC(kernel=best_params['kernel'], C=best_params['C'], gamma=best_params['gamma'])
svc_tuned_model.fit(X_train, y_train)

Now, let's evaluate the performance of the model with the tuned hyperparameters.

In [ ]:
y_pred_tuned = svc_tuned_model.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Accuracy with tuned parameters: {acc_tuned * 100:.2f}%")

con_tuned = confusion_matrix(y_test, y_pred_tuned)
print("\nConfusion Matrix with tuned parameters:\n", con_tuned)

print("\nClassification Report with tuned parameters:\n", classification_report(y_test, y_pred_tuned))

this resulted in a lower overall accuracy and generally worse performance metrics

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt_model.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f"Decision Tree Accuracy: {acc_dt * 100:.2f}%")

con_dt = confusion_matrix(y_test, y_pred_dt)
print("\nDecision Tree Confusion Matrix:\n", con_dt)

print("\nDecision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {acc_rf * 100:.2f}%")

con_rf = confusion_matrix(y_test, y_pred_rf)
print("\nRandom Forest Confusion Matrix:\n", con_rf)

print("\nRandom Forest Classification Report:\n", classification_report(y_test, y_pred_rf))

### Summary and Best Performing Model:

Based on accuracy, the untuned SVC model achieved the highest accuracy of 79.67%. While its precision for class 0 (0.92) is very high, its recall for class 0 (0.33) is quite low, indicating it misses many actual class 0 cases. However, for class 1, it has excellent recall (0.99).

The Random Forest Classifier is also a strong contender with an accuracy of 78.05%, showing a better balance between precision and recall for class 0 compared to the untuned SVC (precision 0.74, recall 0.39) and strong performance for class 1 (precision 0.79, recall 0.94).

